# Surrogate Factory — UCFatigue
## Chapter 9. Model Validation
Objectives:
- **9.0** Validate the train/test split quality (voxel tesselation proximity method).
- **9.1** Predict fatigue loads on the test set.
- **9.2** Compute metrics (R², MAE, quantile90).
- **9.2b** KS distribution tests: train vs test residuals.
- **9.3** Validate against requirements defined in SF_1.
- **9.4** Generate scatter and ratio plots.

### 0. Workflow initialisation

In [ ]:
import sys
from pathlib import Path

# add repo root so validationlib is importable
repo_root = str(Path('..').resolve().parent)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from IPython.display import display, HTML, JSON
from surrogate_factory.workflow import Workflow

workflow = Workflow('pipeline_config.yaml')
workflow.resume()

### 9. Model Validation

In [ ]:
workflow.import_metadata(stage_name='SF_9_Model_Validation')

In [ ]:
Train_set = workflow.load_data(workflow.config['job_name'] + '_Train_set.csv')
Test_set  = workflow.load_data(workflow.config['job_name'] + '_Test_set.csv')
print(f'Train set : {Train_set.shape}')
print(f'Test set  : {Test_set.shape}')

#### 9.0 Split Validation
Checks that the train/test split is statistically sound:
- **Residual voxel proportion**: fraction of test points in category combinations not seen in training (target ≤ 0.05).
- **Phacking**: test points too close to training points (data leakage risk).
- **Isolated test**: test points too far from any training point (extrapolation).
- **Chi² p-value**: distribution of train vs test samples per voxel (target ≥ 0.05).

In [ ]:
from model_validation.split_val import split_validation
split_result = split_validation(workflow, Train_set, Test_set)

#### 9.1 Predictions

In [ ]:
from model_validation.prediction import predict
model_output = predict(workflow, Test_set)
train_output = predict(workflow, Train_set)  # needed for distribution tests
model_output.head()

#### 9.2 Metrics

In [ ]:
from model_validation.score import calculate_metrics
metrics = calculate_metrics(workflow, Test_set, model_output)
JSON(metrics)

#### 9.2b Distribution Tests (KS: train vs test residuals)
Kolmogorov-Smirnov test comparing the residual distributions on the **training set** vs the **test set** for each model and output.
- H₀: both distributions are the same.
- p-value ≥ 0.05 → ✓ no significant difference (healthy generalisation).
- p-value < 0.05 → ✗ residuals differ (possible overfitting or distribution shift).

In [ ]:
from model_validation.score import distribution_tests
ks_results = distribution_tests(workflow, Train_set, Test_set, train_output, model_output)

#### 9.3 Validation against requirements

In [ ]:
from model_validation.validation import validate
validate(workflow, metrics)

#### 9.4 Plots

In [ ]:
%matplotlib inline
from model_validation.visualize import plot
plot(workflow, Test_set, model_output)

### Save

In [ ]:
workflow.save_metadata()